# Incident Response Runbook: Initial Access - Exploit Public-Facing Application

**Tactic:** Initial Access
**Technique:** Exploit Public-Facing Application (T1190)
**Severity:** CRITICAL

## Overview

This runbook provides step-by-step incident response procedures for Exploit Public-Facing Application activities.

## Incident Response Phases

1. **Detection & Analysis**
2. **Containment**
3. **Eradication**
4. **Recovery**
5. **Post-Incident Activities**


## Phase 1: Detection & Analysis

### Objectives
- Confirm the incident
- Identify the scope
- Assess impact


In [ ]:
import json
import re
from datetime import datetime
import sys
import os

# Add project root to path for imports
sys.path.append(os.path.join(os.path.dirname(__file__), '..', '..'))

from splunk.splunk_data_collector import SplunkDataCollector
from crowdstrike.crowdstrike_response import CrowdStrikeResponse
from iris.iris_integration import IRISIntegration
from misp.misp_integration import MISPIntegration
from shuffle.shuffle_integration import ShuffleIntegration

# Initialize integrations
splunk = SplunkDataCollector()
crowdstrike = CrowdStrikeResponse()
iris = IRISIntegration()
misp = MISPIntegration()
shuffle = ShuffleIntegration()

# Step 1: Detection & Analysis
print("=" * 60)
print("STEP 1: Detection & Analysis")
print("=" * 60)

detection_time = datetime.now().isoformat()
affected_systems = []
exploit_indicators = []
incident_id = None

# Query Splunk for public-facing application exploit indicators
print(f"\n[DETECTION] Querying Splunk for public-facing application exploits...")
try:
    # Web application exploit detection
    web_exploit_query = """
    index=* sourcetype=web_access OR sourcetype=iis OR sourcetype=apache OR sourcetype=nginx
    | search ("sql injection" OR "xss" OR "command injection" OR "path traversal" OR "rce" OR "remote code execution")
    | eval risk_score = case(
        match(_raw, "sql.*injection"), 10,
        match(_raw, "remote.*code.*execution"), 10,
        match(_raw, "command.*injection"), 9,
        match(_raw, "path.*traversal"), 8,
        match(_raw, "xss"), 7,
        match(_raw, "exploit"), 8,
        1==1, 6
    )
    | where risk_score >= 7
    | stats count by src_ip, uri, user_agent, status, risk_score
    | sort -risk_score
    """

    web_results = splunk.search(web_exploit_query, timeframe="-24h")
    if web_results:
        print(f"   Found {len(web_results)} web application exploit attempts")
        for result in web_results[:10]:  # Top 10
            exploit_indicators.append({
                'type': 'web_exploit',
                'src_ip': result.get('src_ip'),
                'uri': result.get('uri'),
                'user_agent': result.get('user_agent'),
                'status': result.get('status'),
                'risk_score': result.get('risk_score')
            })

    # API exploit detection
    api_exploit_query = """
    index=* sourcetype=api_access OR sourcetype=web_proxy
    | search ("api.*exploit" OR "graphql.*injection" OR "rest.*attack" OR "soap.*exploit")
    | eval risk_score = case(
        match(_raw, "api.*injection"), 9,
        match(_raw, "graphql.*attack"), 8,
        match(_raw, "soap.*exploit"), 8,
        match(_raw, "rest.*attack"), 7,
        1==1, 6
    )
    | where risk_score >= 7
    | stats count by src_ip, endpoint, method, response_code, risk_score
    | sort -risk_score
    """

    api_results = splunk.search(api_exploit_query, timeframe="-24h")
    if api_results:
        print(f"   Found {len(api_results)} API exploit attempts")
        for result in api_results[:10]:  # Top 10
            exploit_indicators.append({
                'type': 'api_exploit',
                'src_ip': result.get('src_ip'),
                'endpoint': result.get('endpoint'),
                'method': result.get('method'),
                'response_code': result.get('response_code'),
                'risk_score': result.get('risk_score')
            })

    # Application vulnerability exploitation
    vuln_exploit_query = """
    index=* sourcetype=vulnerability_scanner OR sourcetype=waf OR sourcetype=ids
    | search ("vulnerability.*exploited" OR "cve.*exploit" OR "zero.*day" OR "exploit.*attempt")
    | eval risk_score = case(
        match(_raw, "zero.*day"), 10,
        match(_raw, "cve.*critical"), 9,
        match(_raw, "vulnerability.*exploited"), 8,
        match(_raw, "exploit.*success"), 9,
        1==1, 7
    )
    | where risk_score >= 7
    | stats count by src_ip, vulnerability_id, application, risk_score
    | sort -risk_score
    """

    vuln_results = splunk.search(vuln_exploit_query, timeframe="-24h")
    if vuln_results:
        print(f"   Found {len(vuln_results)} vulnerability exploitation attempts")
        for result in vuln_results[:10]:  # Top 10
            exploit_indicators.append({
                'type': 'vulnerability_exploit',
                'src_ip': result.get('src_ip'),
                'vulnerability_id': result.get('vulnerability_id'),
                'application': result.get('application'),
                'risk_score': result.get('risk_score')
            })

except Exception as e:
    print(f"   Splunk query failed: {e}")

# Get unique affected systems/applications
unique_ips = set()
unique_applications = set()
for indicator in exploit_indicators:
    if indicator.get('src_ip'):
        unique_ips.add(indicator['src_ip'])
    if indicator.get('application'):
        unique_applications.add(indicator['application'])

# Check CrowdStrike for related endpoint activity
print(f"\n[DETECTION] Checking CrowdStrike for related endpoint activity...")
for ip in unique_ips:
    try:
        cs_alerts = crowdstrike.get_alerts_by_ip(ip)
        if cs_alerts:
            for alert in cs_alerts[:5]:  # Top 5 alerts per IP
                affected_systems.append({
                    'ip': ip,
                    'alerts': len(cs_alerts),
                    'device_id': alert.get('device_id'),
                    'hostname': alert.get('hostname')
                })
            print(f"   Found {len(cs_alerts)} CrowdStrike alerts for IP {ip}")
    except Exception as e:
        print(f"   CrowdStrike check failed for {ip}: {e}")

# Enrich with threat intelligence
print(f"\n[INTEL] Enriching with threat intelligence...")
for indicator in exploit_indicators[:5]:  # Check top 5
    try:
        if indicator.get('src_ip'):
            ti_data = misp.lookup_ip(indicator['src_ip'])
            if ti_data:
                indicator['threat_intel'] = ti_data
                print(f"   Threat intel found for IP: {indicator['src_ip']}")
        elif indicator.get('vulnerability_id'):
            ti_data = misp.lookup_cve(indicator['vulnerability_id'])
            if ti_data:
                indicator['threat_intel'] = ti_data
                print(f"   Threat intel found for CVE: {indicator['vulnerability_id']}")
    except Exception as e:
        print(f"   Threat intelligence lookup failed: {e}")

# Create IRIS case
print(f"\n[CASE] Creating IRIS incident case...")
try:
    case_data = {
        'title': f'Public-Facing Application Exploit - {len(exploit_indicators)} indicators detected',
        'description': f'Detected exploitation attempts against public-facing applications affecting {len(unique_applications)} applications and {len(unique_ips)} source IPs',
        'severity': 'CRITICAL',
        'tactic': 'Initial Access',
        'technique': 'Exploit Public-Facing Application (T1190)',
        'indicators': exploit_indicators[:10],  # Top 10 indicators
        'detection_time': detection_time
    }
    incident_id = iris.create_case(case_data)
    print(f"   Created IRIS case: {incident_id}")
except Exception as e:
    print(f"   IRIS case creation failed: {e}")

print(f"\n Detection complete:")
print(f"  - {len(exploit_indicators)} exploit indicators detected")
print(f"  - {len(unique_ips)} suspicious source IPs identified")
print(f"  - {len(unique_applications)} affected applications")
print(f"  - Threat intelligence enriched for {len([i for i in exploit_indicators if i.get('threat_intel')])} indicators")
print(f"  - IRIS case created: {incident_id}")


## Phase 2: Containment

### Objectives
- Stop the attack
- Isolate affected systems
- Prevent spread


In [ ]:
# Step 2: Containment
print("\n" + "=" * 60)
print("STEP 2: Containment")
print("=" * 60)

containment_time = datetime.now().isoformat()
isolated_systems = []
blocked_ips = []
blocked_urls = []
disabled_services = []

# Isolate affected systems
print(f"\n[ACTION] Isolating affected systems...")
for system in affected_systems:
    try:
        if system.get('device_id'):
            isolation_result = crowdstrike.isolate_host(system['device_id'])
            if isolation_result.get('status') == 'isolated':
                isolated_systems.append(system.get('hostname', system.get('ip', 'unknown')))
                print(f"   Isolated system: {system.get('hostname', system.get('ip', 'unknown'))}")
    except Exception as e:
        print(f"   System isolation failed for {system.get('hostname', system.get('ip', 'unknown'))}: {e}")

# Block malicious source IPs
print(f"\n[ACTION] Blocking malicious source IPs...")
for ip in unique_ips:
    try:
        block_result = shuffle.block_ip_address(ip)
        if block_result.get('status') == 'blocked':
            blocked_ips.append(ip)
            print(f"   Blocked malicious IP: {ip}")
    except Exception as e:
        print(f"   IP blocking failed for {ip}: {e}")

# Block exploited URLs/endpoints
print(f"\n[ACTION] Blocking exploited URLs and endpoints...")
for indicator in exploit_indicators:
    try:
        if indicator.get('uri'):
            block_result = shuffle.block_url(indicator['uri'])
            if block_result.get('status') == 'blocked':
                blocked_urls.append(indicator['uri'])
                print(f"   Blocked exploited URL: {indicator['uri']}")
        elif indicator.get('endpoint'):
            block_result = shuffle.block_endpoint(indicator['endpoint'])
            if block_result.get('status') == 'blocked':
                blocked_urls.append(indicator['endpoint'])
                print(f"   Blocked exploited endpoint: {indicator['endpoint']}")
    except Exception as e:
        print(f"   URL/endpoint blocking failed: {e}")

# Disable vulnerable services/applications
print(f"\n[ACTION] Disabling vulnerable services and applications...")
for app in unique_applications:
    try:
        disable_result = shuffle.disable_vulnerable_service(app)
        if disable_result.get('status') == 'disabled':
            disabled_services.append(app)
            print(f"   Disabled vulnerable service: {app}")
    except Exception as e:
        print(f"   Service disabling failed for {app}: {e}")

# Enable enhanced web application monitoring
print(f"\n[ACTION] Enabling enhanced web application monitoring...")
monitoring_rules = []
try:
    # Enable CrowdStrike web exploit monitoring
    cs_monitoring = crowdstrike.enable_enhanced_monitoring('web_exploit')
    if cs_monitoring.get('status') == 'enabled':
        monitoring_rules.append('CrowdStrike web exploit monitoring')
        print(f"   Enabled CrowdStrike web exploit monitoring")

    # Enable Splunk web exploit correlation rules
    splunk_monitoring = splunk.enable_correlation_rule('web_exploit')
    if splunk_monitoring.get('status') == 'enabled':
        monitoring_rules.append('Splunk web exploit correlation')
        print(f"   Enabled Splunk web exploit correlation rules")
except Exception as e:
    print(f"   Enhanced monitoring setup failed: {e}")

# Update IRIS case with containment actions
print(f"\n[ACTION] Updating IRIS case with containment actions...")
try:
    containment_summary = {
        'isolated_systems': len(isolated_systems),
        'blocked_ips': len(blocked_ips),
        'blocked_urls': len(blocked_urls),
        'disabled_services': len(disabled_services),
        'monitoring_enabled': len(monitoring_rules),
        'containment_time': containment_time
    }
    iris.update_case(incident_id, 'containment_complete', containment_summary)
    print(f"   Updated IRIS case {incident_id} with containment results")
except Exception as e:
    print(f"   IRIS case update failed: {e}")

print(f"\n Containment complete:")
print(f"  - {len(isolated_systems)} systems isolated")
print(f"  - {len(blocked_ips)} malicious IPs blocked")
print(f"  - {len(blocked_urls)} exploited URLs/endpoints blocked")
print(f"  - {len(disabled_services)} vulnerable services disabled")
print(f"  - {len(monitoring_rules)} enhanced monitoring rules enabled")


## Phase 3: Eradication

### Objectives
- Remove malicious artifacts
- Close vulnerabilities
- Verify threat removal


In [ ]:
# Step 3: Eradication
print("\n" + "=" * 60)
print("STEP 3: Eradication")
print("=" * 60)

eradication_time = datetime.now().isoformat()
terminated_processes = []
removed_payloads = []
patched_vulnerabilities = []
cleaned_logs = []

# Terminate exploit-related processes
print(f"\n[ACTION] Terminating exploit-related processes...")
for system in affected_systems:
    try:
        if system.get('device_id'):
            processes = crowdstrike.get_suspicious_processes(system['device_id'], 'web_exploit')
            for proc in processes:
                if proc.get('process_name') in ['exploit.exe', 'webshell.php', 'malware.dll', 'backdoor.exe']:
                    term_result = crowdstrike.terminate_process(system['device_id'], proc['process_id'])
                    if term_result.get('status') == 'terminated':
                        terminated_processes.append(f"{system.get('hostname', 'unknown')}:{proc['process_name']}")
                        print(f"   Terminated process: {proc['process_name']} on {system.get('hostname', 'unknown')}")
    except Exception as e:
        print(f"   Process termination failed: {e}")

# Remove web shells and exploit payloads
print(f"\n[ACTION] Removing web shells and exploit payloads...")
for system in affected_systems:
    try:
        if system.get('device_id'):
            payloads = crowdstrike.get_web_payloads(system['device_id'])
            for payload_path in payloads:
                remove_result = crowdstrike.remove_file(system['device_id'], payload_path)
                if remove_result.get('status') == 'removed':
                    removed_payloads.append(f"{system.get('hostname', 'unknown')}:{payload_path}")
                    print(f"   Removed web payload: {payload_path} from {system.get('hostname', 'unknown')}")
    except Exception as e:
        print(f"   Payload removal failed for {system.get('hostname', 'unknown')}: {e}")

# Patch exploited vulnerabilities
print(f"\n[ACTION] Patching exploited vulnerabilities...")
for indicator in exploit_indicators:
    try:
        if indicator.get('vulnerability_id'):
            patch_result = shuffle.apply_security_patch(indicator['vulnerability_id'])
            if patch_result.get('status') == 'patched':
                patched_vulnerabilities.append(indicator['vulnerability_id'])
                print(f"   Patched vulnerability: {indicator['vulnerability_id']}")
    except Exception as e:
        print(f"   Vulnerability patching failed for {indicator.get('vulnerability_id')}: {e}")

# Clean exploit-related logs and artifacts
print(f"\n[ACTION] Cleaning exploit-related logs and artifacts...")
try:
    # Clean web server logs
    log_cleanup = splunk.clean_exploit_logs()
    if log_cleanup.get('status') == 'cleaned':
        cleaned_logs.append(f"Web logs cleaned: {log_cleanup.get('count', 0)} entries")
        print(f"   Cleaned up {log_cleanup.get('count', 0)} exploit-related log entries")

    # Clean temporary files and caches
    cache_cleanup = shuffle.clean_web_cache()
    if cache_cleanup.get('status') == 'cleaned':
        cleaned_logs.append(f"Cache cleaned: {cache_cleanup.get('count', 0)} files")
        print(f"   Cleaned up {cache_cleanup.get('count', 0)} cached files")
except Exception as e:
    print(f"   Log cleanup failed: {e}")

# Update threat intelligence
print(f"\n[ACTION] Updating threat intelligence...")
try:
    eradication_iocs = {
        'exploit_ips': list(unique_ips),
        'exploited_vulnerabilities': [i.get('vulnerability_id') for i in exploit_indicators if i.get('vulnerability_id')],
        'exploit_payloads': removed_payloads,
        'affected_applications': list(unique_applications)
    }
    misp.share_iocs(eradication_iocs, 'web_exploit_eradicated')
    print(f"   Shared eradication IOCs with MISP threat intelligence")
except Exception as e:
    print(f"   Threat intelligence update failed: {e}")

# Update IRIS case with eradication actions
print(f"\n[ACTION] Updating IRIS case with eradication actions...")
try:
    eradication_summary = {
        'terminated_processes': len(terminated_processes),
        'removed_payloads': len(removed_payloads),
        'patched_vulnerabilities': len(patched_vulnerabilities),
        'cleaned_logs': len(cleaned_logs),
        'eradication_time': eradication_time
    }
    iris.update_case(incident_id, 'eradication_complete', eradication_summary)
    print(f"   Updated IRIS case {incident_id} with eradication results")
except Exception as e:
    print(f"   IRIS case update failed: {e}")

print(f"\n Eradication complete:")
print(f"  - {len(terminated_processes)} malicious processes terminated")
print(f"  - {len(removed_payloads)} web payloads removed")
print(f"  - {len(patched_vulnerabilities)} vulnerabilities patched")
print(f"  - {len(cleaned_logs)} logs and artifacts cleaned")


## Phase 4: Recovery

### Objectives
- Restore systems
- Validate functionality
- Monitor for issues


In [ ]:
# Step 4: Recovery
print("\n" + "=" * 60)
print("STEP 4: Recovery")
print("=" * 60)

recovery_time = datetime.now().isoformat()
restored_systems = []
restored_services = []
validated_applications = []

# Restore isolated systems
print(f"\n[ACTION] Restoring isolated systems...")
for system in affected_systems:
    try:
        if system.get('device_id') and system.get('hostname') in isolated_systems:
            restore_result = crowdstrike.restore_host(system['device_id'])
            if restore_result.get('status') == 'restored':
                restored_systems.append(system['hostname'])
                print(f"   Restored system: {system['hostname']}")
    except Exception as e:
        print(f"   System restoration failed for {system.get('hostname', 'unknown')}: {e}")

# Restore disabled services and applications
print(f"\n[ACTION] Restoring disabled services and applications...")
for service in disabled_services:
    try:
        restore_result = shuffle.restore_vulnerable_service(service)
        if restore_result.get('status') == 'restored':
            restored_services.append(service)
            print(f"   Restored service: {service}")
    except Exception as e:
        print(f"   Service restoration failed for {service}: {e}")

# Validate application functionality
print(f"\n[ACTION] Validating application functionality...")
for app in unique_applications:
    try:
        validation_result = shuffle.validate_application_integrity(app)
        if validation_result.get('status') == 'valid':
            validated_applications.append(app)
            print(f"   Application integrity validated: {app}")
        else:
            print(f"   Application validation issues detected: {app}")
    except Exception as e:
        print(f"   Application validation failed for {app}: {e}")

# Restore web application services
print(f"\n[ACTION] Restoring web application services...")
try:
    # Restore web server services
    web_restore = shuffle.restore_web_services()
    if web_restore.get('status') == 'restored':
        restored_services.append('Web services')
        print(f"   Restored web application services")

    # Restore API services
    api_restore = shuffle.restore_api_services()
    if api_restore.get('status') == 'restored':
        restored_services.append('API services')
        print(f"   Restored API services")
except Exception as e:
    print(f"   Service restoration failed: {e}")

# Re-enable standard monitoring
print(f"\n[ACTION] Re-enabling standard monitoring...")
try:
    # Restore normal CrowdStrike monitoring
    cs_restore = crowdstrike.restore_standard_monitoring()
    if cs_restore.get('status') == 'restored':
        print(f"   Restored standard CrowdStrike monitoring")

    # Restore normal Splunk correlation rules
    splunk_restore = splunk.restore_standard_correlation()
    if splunk_restore.get('status') == 'restored':
        print(f"   Restored standard Splunk correlation rules")
except Exception as e:
    print(f"   Monitoring restoration failed: {e}")

# Update IRIS case with recovery actions
print(f"\n[ACTION] Updating IRIS case with recovery actions...")
try:
    recovery_summary = {
        'restored_systems': len(restored_systems),
        'restored_services': len(restored_services),
        'validated_applications': len(validated_applications),
        'recovery_time': recovery_time
    }
    iris.update_case(incident_id, 'recovery_complete', recovery_summary)
    print(f"   Updated IRIS case {incident_id} with recovery results")
except Exception as e:
    print(f"   IRIS case update failed: {e}")

print(f"\n Recovery complete:")
print(f"  - {len(restored_systems)} systems restored")
print(f"  - {len(restored_services)} services restored")
print(f"  - {len(validated_applications)} applications validated")


## Phase 5: Post-Incident Activities

### Objectives
- Document findings
- Implement improvements
- Share lessons learned


In [ ]:
# Step 5: Post-Incident
print("\n" + "=" * 60)
print("STEP 5: Post-Incident")
print("=" * 60)

post_incident_time = datetime.now().isoformat()

# Generate incident report
print(f"\n[ACTION] Generating incident report...")
try:
    incident_report = {
        'incident_id': incident_id,
        'technique': 'Exploit Public-Facing Application (T1190)',
        'detection_time': detection_time,
        'containment_time': containment_time,
        'eradication_time': eradication_time,
        'recovery_time': recovery_time,
        'post_incident_time': post_incident_time,
        'affected_systems': len(affected_systems),
        'exploit_indicators': len(exploit_indicators),
        'unique_ips': len(unique_ips),
        'unique_applications': len(unique_applications),
        'isolated_systems': len(isolated_systems),
        'blocked_ips': len(blocked_ips),
        'blocked_urls': len(blocked_urls),
        'disabled_services': len(disabled_services),
        'terminated_processes': len(terminated_processes),
        'removed_payloads': len(removed_payloads),
        'patched_vulnerabilities': len(patched_vulnerabilities),
        'cleaned_logs': len(cleaned_logs),
        'restored_systems': len(restored_systems),
        'restored_services': len(restored_services),
        'validated_applications': len(validated_applications),
        'threat_intelligence_shared': True,
        'lessons_learned': [
            'Enhanced web application vulnerability scanning needed',
            'Improved WAF rules and monitoring required',
            'Regular security patching of public-facing applications',
            'API security testing and validation needed',
            'Web application firewall tuning required'
        ]
    }

    # Save report to IRIS
    iris.save_incident_report(incident_id, incident_report)
    print(f"   Incident report saved to IRIS case {incident_id}")

    # Share anonymized report with MISP
    misp.share_incident_report(incident_report, 'web_exploit_response')
    print(f"   Anonymized incident report shared with MISP community")
except Exception as e:
    print(f"   Report generation failed: {e}")

# Conduct lessons learned session
print(f"\n[ACTION] Conducting lessons learned analysis...")
try:
    lessons_learned = {
        'what_went_well': [
            'Rapid detection of web application exploits',
            'Effective isolation and payload removal',
            'Comprehensive vulnerability patching',
            'Good integration between security tools'
        ],
        'what_could_improve': [
            'Earlier detection of zero-day exploits',
            'Better web application monitoring and alerting',
            'Enhanced threat intelligence for exploit campaigns',
            'More granular access controls for web applications',
            'Automated security testing integration'
        ],
        'preventive_measures': [
            'Implement comprehensive web application firewall',
            'Deploy runtime application self-protection (RASP)',
            'Regular vulnerability scanning and penetration testing',
            'Implement secure coding practices and training',
            'Deploy API security gateways and monitoring',
            'Regular security assessments of public-facing applications'
        ]
    }

    iris.add_lessons_learned(incident_id, lessons_learned)
    print(f"   Lessons learned documented in IRIS case {incident_id}")
except Exception as e:
    print(f"   Lessons learned analysis failed: {e}")

# Update security controls
print(f"\n[ACTION] Updating security controls...")
try:
    # Update Splunk correlation rules
    splunk.update_exploit_rules(incident_report)
    print(f"   Updated Splunk web exploit detection rules")

    # Update CrowdStrike policies
    crowdstrike.update_exploit_policies(incident_report)
    print(f"   Updated CrowdStrike exploit prevention policies")

    # Update Shuffle workflows
    shuffle.update_exploit_workflows(incident_report)
    print(f"   Updated Shuffle exploit response workflows")
except Exception as e:
    print(f"   Security controls update failed: {e}")

# Close incident
print(f"\n[ACTION] Closing incident...")
try:
    closure_summary = {
        'closure_time': post_incident_time,
        'incident_status': 'resolved',
        'follow_up_actions': [
            'Monitor for similar web application exploits',
            'Conduct web application security assessment',
            'Review and update WAF rules',
            'Schedule penetration testing exercises'
        ],
        'metrics': {
            'time_to_detect': 'Calculated from detection_time',
            'time_to_contain': 'Calculated from containment_time',
            'time_to_eradicate': 'Calculated from eradication_time',
            'time_to_recover': 'Calculated from recovery_time',
            'affected_assets': len(affected_systems)
        }
    }

    iris.close_case(incident_id, closure_summary)
    print(f"   IRIS case {incident_id} closed successfully")
except Exception as e:
    print(f"   Incident closure failed: {e}")

print(f"\n Post-incident activities complete:")
print(f"  - Incident report generated and shared")
print(f"  - Lessons learned documented")
print(f"  - Security controls updated")
print(f"  - Incident case closed")

print(f"\n Public-facing application exploit incident response completed successfully!")
print(f"   Incident ID: {incident_id}")
print(f"   Total duration: {len(exploit_indicators)} exploit indicators mitigated")


## Summary

This incident response runbook has guided you through the complete lifecycle of responding to this incident.

### Key Takeaways
- Rapid detection and response are critical
- Containment prevents further damage
- Thorough eradication prevents reinfection
- Continuous monitoring ensures recovery

### Next Steps
- Review and approve the incident report
- Implement recommended preventive measures
- Schedule follow-up review
- Share lessons learned with the team
